# 🏷️ Мониторинг цен конкурентов на красную икру

**Этот ноутбук собирает цены с трёх сайтов и показывает сравнение.**

Нажми `Runtime` → `Run all` (или `Среда выполнения` → `Выполнить всё`) — и всё запустится автоматически.

---

### Что делает этот ноутбук

1. Устанавливает Python-библиотеки (httpx, lxml, playwright)
2. Клонирует проект с GitHub
3. Запускает сбор цен с трёх сайтов конкурентов
4. Выводит таблицу сравнения

### Отслеживаемые сайты

| Сайт | Метод сбора |
|---|---|
| apeti.ru | Парсинг HTML |
| seafood-shop.ru (Икорный) | JSON API |
| delikateska.ru (Деликатеска) | GraphQL API |

### Наш продукт

**Икра горбуши 120 г** — базовая цена 1 590 ₽ (можно изменить в `data/user_price.json`)

## Шаг 1: Установка зависимостей

Устанавливаем библиотеки, нужные для сбора данных с сайтов.

In [ ]:
# ============================================================
# ШАГ 1: Установка Python-пакетов
# ============================================================

# httpx — библиотека для HTTP-запросов (как requests, но быстрее)
# Нужна чтобы скачивать страницы сайтов и вызывать API
!pip install -q httpx

# lxml — библиотека для разбора HTML
# Нужна чтобы вытаскивать названия и цены из HTML-кода страниц
!pip install -q lxml

# playwright — браузерная автоматизация
# Нужна для сайта delikateska.ru (запасной вариант, если API не ответит)
# -q значит "тихий режим" — меньше вывода в лог
!pip install -q playwright

# Устанавливаем браузер Chromium для Playwright
# Это загружает ~120 МБ один раз
!playwright install chromium 2>&1 | tail -3

print("✅ Зависимости установлены")

## Шаг 2: Загрузка проекта

Клонируем проект с GitHub — это наш код для сбора цен.

In [ ]:
# ============================================================
# ШАГ 2: Клонирование проекта с GitHub
# ============================================================

# git clone — команда для скачивания репозитория
# https://github.com/tswtim/price-monitor.git — адрес твоего репозитория
# Весь код (адаптеры, нормализация, отчёты) лежит в этом репозитории
!git clone -q https://github.com/tswtim/price-monitor.git

# %cd — магическая команда Jupyter: перейти в папку проекта
# Все дальнейшие команды будут выполняться внутри price-monitor/
%cd price-monitor

print("✅ Проект загружен")

## Шаг 3: Настройка нашей цены

Создаём файл с нашей ценой для сравнения с конкурентами. Измени число на актуальную цену.

In [ ]:
# ============================================================
# ШАГ 3: Установка нашей цены
# ============================================================

# json — встроенный модуль Python для работы с JSON-файлами
import json

# Наша цена в рублях за 120 г икры горбуши
# ПОМЕНЯЙ ЭТО ЧИСЛО на свою актуальную цену!
our_price = {
    "price_rub": 1590  # <-- цена за 1 банку 120 г
}

# Записываем цену в JSON-файл, который читает скрипт мониторинга
# data/user_price.json — стандартный путь, куда скрипт смотрит при запуске
with open("data/user_price.json", "w", encoding="utf-8") as f:
    json.dump(our_price, f, ensure_ascii=False, indent=2)

print(f"✅ Наша цена установлена: {our_price['price_rub']:,} ₽ за 120 г".replace(",", " "))

## Шаг 4: Пояснение — как устроен сбор данных

Перед запуском посмотрим, из чего состоит проект.

In [ ]:
# ============================================================
# ШАГ 4: Обзор структуры проекта
# ============================================================

# Показываем, какие файлы есть в проекте
# Это просто для понимания — можно пропустить

print("📁 Структура проекта:\n")
print("monitor/")
print("  ├── cli.py              ← точка входа, команды --run --report")
print("  ├── config.py           ← настройки: сайты, товары, цены")
print("  ├── normalize.py        ← извлечение веса, расчёт цены за 100г")
print("  ├── store.py            ← сохранение истории в SQLite")
print("  ├── report.py           ← форматирование таблиц, CSV, JSON")
print("  └── adapters/           ← сборщики для каждого сайта")
print("       ├── apeti.py       ← apeti.ru (парсинг HTML)")
print("       ├── seafood_shop.py← seafood-shop.ru (JSON API)")
print("       └── delikateska.py ← delikateska.ru (GraphQL API)")
print("")
print("data/")
print("  ├── prices.db           ← база SQLite с историей цен")
print("  ├── user_price.json     ← наша цена (создали выше)")
print("  ├── reports/            ← CSV и JSON отчёты")
print("  └── snapshots/          ← сырые данные с сайтов")

print("\n💡 Как это работает:")
print("  1. cli.py запускает все три адаптера")
print("  2. Каждый адаптер идёт на свой сайт и собирает товары")
print("  3. normalize.py фильтрует: оставляет только красную икру")
print("  4. normalizer.py считает цену за 100 г для сравнения")
print("  5. report.py выводит таблицу")
print("  6. store.py сохраняет всё в базу для истории")

## Шаг 5: Запуск сбора цен

Основной шаг — запускаем сбор со всех трёх сайтов и получаем таблицу.

In [ ]:
# ============================================================
# ШАГ 5: ЗАПУСК СБОРА ЦЕН (основной шаг)
# ============================================================

# python -m monitor.cli — запускает модуль monitor/cli.py
# --run — команда «собрать данные со всех сайтов и показать отчёт»
# 
# Под капотом происходит:
#   1. apeti.ru       → парсинг HTML, 155 товаров → фильтр → ~7 красной икры
#   2. seafood-shop.ru → JSON API, 32 товара → фильтр → ~32 красной икры
#   3. delikateska.ru  → GraphQL API, 48 товаров → фильтр → ~7 красной икры
#   4. normalizer.py   → оставляет только «горбушу», считает цену за 100 г
#   5. report.py       → печатает таблицу
#   6. store.py        → сохраняет в data/prices.db

!python -m monitor.cli --run

## Шаг 6: Сохранённые файлы

Проверим, какие отчёты сохранились после запуска.

In [ ]:
# ============================================================
# ШАГ 6: Просмотр сохранённых отчётов
# ============================================================

import os

# Показываем CSV-отчёты (можно открыть в Excel)
print("📁 CSV-отчёты (для Excel):")
csv_files = !ls data/reports/*.csv 2>/dev/null
for f in csv_files:
    size = os.path.getsize(f)
    print(f"   {f} ({size} байт)")

print("\n📁 JSON-отчёты (для программной обработки):")
json_files = !ls data/reports/*.json 2>/dev/null
for f in json_files:
    size = os.path.getsize(f)
    print(f"   {f} ({size:,} байт)".replace(",", " "))

print("\n📁 База данных SQLite:")
db_exists = os.path.exists("data/prices.db")
if db_exists:
    db_size = os.path.getsize("data/prices.db")
    print(f"   data/prices.db ({db_size:,} байт)".replace(",", " "))
    print("   В этой базе хранится ВСЯ история цен")
    print("   При следующем запуске скрипт сравнит новые цены со старыми")

## Шаг 7 (бонус): История цен

При повторных запусках можно посмотреть, как менялись цены.

In [ ]:
# ============================================================
# ШАГ 7: История изменений цен
# ============================================================

# --history показывает динамику цен из базы данных
# Если запускаешь в первый раз — покажет «Нет сохранённых данных»,
# потому что история накапливается после нескольких запусков

!python -m monitor.cli --history

---

## 🎯 Итог

**Что ты получил:**
- Таблицу цен конкурентов (нормализованных к цене за 100 г)
- Сравнение своей цены с рынком
- CSV-файл для Excel
- Базу данных с историей (накапливается при повторных запусках)

**Что делать дальше:**
- Поменяй цену в Шаге 3 на актуальную
- Запускай ноутбук раз в неделю — следи за рынком
- При повторном запуске появятся дельты изменений (⬆/⬇)

**Где взять код:** https://github.com/tswtim/price-monitor